# Managing Message State & History in LangChain
This notebook covers how to use LangChain's message schemas (`SystemMessage`, `HumanMessage`, `AIMessage`, and `ToolMessage`) to direct model behavior and build multi-turn chat applications.

### Step 1: Initialize Chat Model

In [1]:
import os
from langchain_groq import ChatGroq

# Ensure the API key is in the environment
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize model
groq_model = ChatGroq(model="qwen/qwen3-32b")

### Step 2: Simple String Invocation
Under the hood, passing a string is automatically wrapped in a `HumanMessage`.

In [2]:
response = groq_model.invoke("Tell me about the rules of badminton game.")
print(response.content[:500] + "...")

<think>
Okay, so I need to explain the rules of badminton. Let me start by recalling what I know. Badminton is a racket sport played with a shuttlecock. It can be played either in singles or doubles. The goal is to hit the shuttlecock over the net and into the opponent's court in such a way that they can't return it.

First, the court. I think the court dimensions are different for singles and doubles. For singles, the court is narrower, and in doubles, the sides are wider. The net is 1.55 meter...


### Step 3: Structured Messages (System & Human Messages)
We use `SystemMessage` to set instructions or context for the assistant, and `HumanMessage` for user queries.

In [3]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# Define the messages array
messages = [
    SystemMessage("You are a poetry expert."),
    HumanMessage("Write a poem on the name TEJA .")
]

# Get the response from model
response = groq_model.invoke(messages)
response.content

'<think>\nOkay, the user wants a poem about the name Teja. Let me start by breaking down the name. Teja is of Sanskrit origin, meaning "radiance" or "splendor." It\'s also a common name in Indian cultures, both for males and females. I should incorporate that meaning into the poem.\n\nFirst, I need to think of imagery related to radiance—like sunlight, fire, stars. Maybe use metaphors that convey brightness and energy. Also, since it\'s a name poem, it should be personal, maybe addressing Teja directly. \n\nI should consider the structure. Maybe four-line stanzas with a rhyme scheme. Let\'s go with a simple ABAB rhyme scheme for flow. Each stanza can highlight a different aspect of the name\'s meaning. \n\nStart with the name itself, then move to natural elements like dawn, flames, stars. Maybe include cultural references, like Sanskrit roots or references to deities associated with light, such as Surya, the sun god. \n\nI should also make sure the poem feels personal and heartfelt. Pe

### Step 4: Multi-Turn Conversation History
By passing previous `AIMessage` responses back with new user inputs, the model maintains full context of the conversation.

In [4]:
ai_msg = AIMessage("I'd be happy to help you with that.")

# Build conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Great! What is the factorial of 5 ?")
]

# Get final response
response = groq_model.invoke(messages)
print(response.content)

<think>
Okay, the user asked for the factorial of 5. Let me recall what a factorial is. The factorial of a number n, denoted as n!, is the product of all positive integers up to n. So for 5, it's 5 × 4 × 3 × 2 × 1. Let me calculate that step by step. 5 × 4 is 20, then 20 × 3 is 60, 60 × 2 is 120, and 120 × 1 is 120. So the answer should be 120. I should make sure I didn't skip any steps or multiply wrong numbers. Let me check again: 5×4=20, 20×3=60, 60×2=120, 120×1=120. Yep, that's correct. The user might be learning math basics, maybe in school or just for personal knowledge. They might need this for homework or a programming problem. I should explain the steps clearly so they can understand the process, not just the answer. Also, maybe mention that factorials are used in permutations and combinations. Alright, time to put it all together in a friendly way.
</think>

The factorial of 5 (written as **5!**) is calculated as:

**5! = 5 × 4 × 3 × 2 × 1 = 120**

Let me know if you'd like h

### Step 5: Check Usage Metadata (Tokens)

In [5]:
# Inspect input/output token counts
response.usage_metadata

{'input_tokens': 53, 'output_tokens': 327, 'total_tokens': 380}

### Step 6: Managing History Limits (Context Truncation / Pruning)
As conversation histories grow, they can exceed context window limits or increase API costs. A common practice is trimming the history before invoking the model while keeping the main instructions (`SystemMessage`) intact.

In [13]:
def trim_history(messages_list, max_turns=2):
    """
    Keeps the SystemMessage(s) at the start, and retains only 
    the last `max_turns * 2` messages of conversational interaction.
    """
    system_messages = [msg for msg in messages_list if isinstance(msg, SystemMessage)]
    chat_messages = [msg for msg in messages_list if not isinstance(msg, SystemMessage)]
    
    # Keep only the last N chat messages (human/AI alternates)
    trimmed_chat = chat_messages[-(max_turns * 2):]
    
    return system_messages + trimmed_chat

# Create a simulated long conversation
full_chat = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("My favorite color is green."),
    AIMessage("That's a nice color!"),
    HumanMessage("I live in Seattle."),
    AIMessage("Cool! Seattle is a beautiful city."),
    HumanMessage("What is my favorite color?")
]

print(f"Original Chat Length: {len(full_chat)} messages.")

# Trim to only keep the last 1 turn (last 2 messages of chat) + the system prompt
trimmed_chat = trim_history(full_chat, max_turns=1)
print(f"Trimmed Chat Length: {len(trimmed_chat)} messages.")
print("\nTrimmed Chat Context:")
for msg in trimmed_chat:
    print(f"[{type(msg).__name__}]: {msg.content}")

Original Chat Length: 6 messages.
Trimmed Chat Length: 3 messages.

Trimmed Chat Context:
[SystemMessage]: You are a helpful assistant.
[AIMessage]: Cool! Seattle is a beautiful city.
[HumanMessage]: What is my favorite color?


### Step 7: Complete Message flow in Tool Calling (Visualization)
Here we show a complete sequence of LangChain message states that occur during a single tool call request and response loop.

In [7]:
# A mock sequence representing how LangGraph or LangChain executes a tool-calling flow:
tool_flow_messages = [
    SystemMessage("You are an assistant with access to weather tools."),
    
    # 1. User message
    HumanMessage("Is it raining in London?"),
    
    # 2. Assistant message indicating a tool call is needed
    AIMessage(
        content="", 
        tool_calls=[{
            "name": "get_weather", 
            "args": {"location": "London"}, 
            "id": "call_ldn_123", 
            "type": "tool_call"
        }]
    ),
    
    # 3. Tool response message containing the outcome of the Python function execution
    ToolMessage(
        content="It is currently raining heavily in London with a temperature of 14°C.",
        name="get_weather",
        tool_call_id="call_ldn_123"
    )
]

print("Pre-constructed messages for tool-use verification:")
for idx, msg in enumerate(tool_flow_messages):
    print(f"Step {idx}: [{type(msg).__name__}] -> Content: '{msg.content}'")

Pre-constructed messages for tool-use verification:
Step 0: [SystemMessage] -> Content: 'You are an assistant with access to weather tools.'
Step 1: [HumanMessage] -> Content: 'Is it raining in London?'
Step 2: [AIMessage] -> Content: ''
Step 3: [ToolMessage] -> Content: 'It is currently raining heavily in London with a temperature of 14°C.'
